# Original Transolver — same-case step-flow comparison

FlowMLLab · Ehsan Roohi · g011, Re = 25 / 50 / 100

This notebook reads **actual trained checkpoints and predictions** for the original
Transolver. It is distinct from GeoTransolver in the double-step family study.
The exact historical split is 106 training, 21 validation and three test cases.
All fields are from the existing OpenFOAM archive, not the rarefied DSMC archive.

Three new Transolver seeds are compared with the retained seed-17 Geo-DeepONet,
FNO and U-FNO fields. All plots use seed 17 fixed in advance. Different losses,
training point exposures and hardware prevent a controlled architecture ranking.

Run in a full repository checkout with `pip install -e '.[transolver,test]'`
and `pip install ipykernel`. Training is optional: checkpoints, scalers and raw arrays are included with explicit author authorization.
See [full results and protocol](../../results/step_transolver/README.md).

Execution note: cells were checked sequentially in-process with IPython; a separate Jupyter kernel could not start because this execution environment prohibits socket binding.

In [1]:
from pathlib import Path
import sys, subprocess, json
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'qa/verify_step_transolver.py').is_file())
sys.path.insert(0, str(ROOT/'qa'))
OUT=ROOT/'results/step_transolver'
if all((OUT/f'seed_{seed}/best.pt').exists() for seed in (17,29,43)):
    subprocess.run([sys.executable, str(ROOT/'qa/verify_step_transolver.py')], check=True, capture_output=True)
    subprocess.run([sys.executable, str(ROOT/'qa/plot_step_transolver.py')], check=True, capture_output=True)
    print('Verified all nine saved predictions from local checkpoints and recomputed metrics.')
else:
    print('Viewing committed results. Run qa/train_step_transolver.py for seeds 17/29/43 to regenerate raw artifacts before independent verification.')

Verified all nine saved predictions from local checkpoints and recomputed metrics.


## Results and uncertainty

Velocity and centered-pressure errors are relative L2 percentages. Reverse IoU
uses **all fluid cells with u < 0**, not the filtered post-audit metric.
Means and sample standard deviations below concern three Transolver seeds only;
historical models have one retained seed. Checkpoints use validation only, and
all three selected checkpoints are at the 400-epoch budget boundary, so training
saturation is not established.

In [2]:
display(pd.read_csv(OUT/'comparison.csv').round(4))
display(pd.DataFrame(json.loads((OUT/'summary.json').read_text())).round(4))

,Re,model,seed,velocity_percent,pressure_percent,reverse_iou,reverse_area_bias_percent
0,25,Geo-DeepONet,17,29.8898,47.8743,0.3994,5.5872
1,25,FNO,17,17.3778,52.3667,0.6642,0.6637
2,25,U-FNO,17,19.4525,48.4315,0.6582,0.8096
3,25,Transolver,17,23.9910,70.5154,0.3343,-2.0861
4,25,Transolver,29,27.6163,72.2669,0.4674,-1.9183
5,25,Transolver,43,25.3894,71.3493,0.6217,-1.1597
6,50,Geo-DeepONet,17,26.1111,32.3959,0.6030,3.0926
7,50,FNO,17,20.2189,47.4795,0.6388,0.8607
8,50,U-FNO,17,21.5049,72.1918,0.6721,0.7877
9,50,Transolver,17,27.8670,91.0255,0.6035,1.3786


,Re,velocity_percent_mean,velocity_percent_std,pressure_percent_mean,pressure_percent_std,reverse_iou_mean,reverse_iou_std
0,25,25.6655,1.8284,71.3772,0.8761,0.4744,0.1438
1,50,28.6042,0.8872,93.3744,11.4525,0.6407,0.0371
2,100,36.4342,1.5667,73.0785,1.6974,0.5678,0.0861


## Field comparisons

CFD / Geo-DeepONet / FNO / U-FNO / original Transolver. All neural rows show
seed 17, not a test-selected seed. Shared scales, actual per-row streamlines,
physical 5:1 aspect and identical reference fields. Pressure means are removed
independently. Vorticity excludes stencils crossing solid boundaries.

Speed is shown at full width for GitHub readability. Separate pressure and vorticity comparisons are available in the results page.

In [3]:
for re in (100,50,25):
    display(Image(filename=str(OUT/f'speed_Re{re}.png')))

### Committed figures (linked rather than duplicated in notebook outputs)

![Re 100 comparison](../../results/step_transolver/all_models_g011_Re100.png)

![Re 50 comparison](../../results/step_transolver/speed_Re50.png)

![Re 25 comparison](../../results/step_transolver/speed_Re25.png)

## What this run establishes

Original Transolver improves mean velocity error relative to the historical
Geo-DeepONet run at Re=25, but worsens it at Re=50/100. Centered-pressure errors
are worse at all three Reynolds numbers. This compact run does **not** support
replacing the existing model as a generally superior predictor.

The full contextual fluid cloud is used at inference; chunking would change
slice aggregation. Geometry-level separation and train-only scaling prevent
label leakage, but this historically inspected geometry is not a fresh blind test.
Training uses uniform fluid sampling and unweighted normalized MSE; historical
models used a different weighted loss. These differences remain visible in the
protocol rather than being presented as a matched optimization study.

Reproduce training with `python qa/train_step_transolver.py --seed 17` (and
29/43). Source, weights, histories, scalers, raw fields and hashes are included. No new CFD is needed.